<a href="https://colab.research.google.com/github/Pro5507/ABS-and-Polly/blob/main/Math_Mastermind.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import config
from openai import OpenAI

GROQ_URL = "https://api.groq.com/openai/v1"
MODELS = getattr(config, "GROQ_MODELS", ["llama-3.1-8b-instant", "mixtral-8x7b-32768"])

def generate_response(prompt: str, temperature: float = 0.3, max_tokens: int = 512) -> str:
  key = getattr(config, "GROQ_API_KEY", None)
  if not key:
    return "Error: GROQ_API_KEY missing in config.py"
  c = OpenAI(api_key=key, base_url=GROQ_URL)
  last_err = None
  for m in MODELS:
    try:
      r = c.chat.completions.create(
          model=m,
          message=[{"role": "user", "content": prompt}],
          temperature=temperature,
          max_tokens=max_tokens,
      )
      return r.choices[0].message.content
    except Exception as e:
      last_err = e
  return (
      "Groq model failed.\n"
      f"Triend models: {MODELS}\n"
      "Fix:\n"
      "1) Switch to hf by importing hf.py in main.py OR\n"
      "2) Replace Groq model in groq.py (GROQ_MODELS).\n"
      f"Details: {type(last_err).__name__}: {last_err}"
  )

In [ ]:
import config
from huggingface_hub import InferenceClient

MODELS = getattr(
    config,
    "HF_MODELS",
    ["meta-llama/Llama-3.1-8B-Instruct"],
)

def generate_response(prompt: str, temperature: float = 0.3, max_tokens: int = 512) -> str:
  key = getattr(config, "HF_API_KEY", None)
  if not key:
    return "Error: HF_API_KEY missing in config.py"
  last_err = None
  for m in MODELS:
    try:
      c = InferenceClient(model=m, token=key)
      r = c.chat_completion(
          message=[{"role": "user", "content": prompt}],
          temperature=temperature,
          max_tokens=max_tokekns,
      )
      return r.choices[0].message.content
    except Exception as e:
      last_err = e
  return (
      "Hugging Face model failed.\n"
      f"Tried models: {MODELS}\n"
      "Fix:\n"
      "1) Switch to Groq by importing groq.py in main.py Or\n"
      "2) Replace HF model in hf.py (HF_MODELS).\n"
      f"Details: {type(last_err).__name__}: {last_err}"
  )